# Step 5: Generate Articles and Quizzes

This notebook takes the raw transcript JSON files and uses an LLM to produce two things per video: a professionally written article summarizing the content, and a multiple-choice quiz for learner assessment.

**What it does:**
1. Reads each transcript JSON (the full Arabic text)
2. Sends the text to an LLM to generate a clean English article with key takeaways and estimated reading time
3. Sends the article + original transcript sentences to the LLM to generate 5-10 multiple-choice quiz questions with per-option feedback
4. Saves both as JSON files alongside the original transcripts

**Input:** Transcript JSON files from Step 2  
**Output:** `*_article.json` and `*_quiz.json` files per video

---
## Prerequisites
- Transcript JSON files from `02_transcribe_asr.ipynb`
- An LLM API key (same setup as Step 3 — any OpenAI-compatible provider works)

> **Note:** This step runs on the original transcript JSONs (not the chunking plans from Step 3). It can run in parallel with or after Steps 3-4.


## Configuration

In [ ]:
import os
import json
import glob
from openai import OpenAI
from google.colab import drive

drive.mount('/content/drive')

# ===============================================
# CONFIGURATION - Edit these values
# ===============================================
INPUT_FOLDER = "/content/drive/MyDrive/your_transcripts/Day_1"
OUTPUT_FOLDER = INPUT_FOLDER  # Articles and quizzes are saved alongside transcripts

# LLM Provider (OpenAI-compatible endpoint)
MODEL_NAME = "kimi-k2-thinking"
API_KEY = ""
BASE_URL = "https://api.moonshot.ai/v1"
# ===============================================

assert API_KEY, "Please set your LLM API key above."

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print("Configuration loaded.")


## System Prompts

Two separate prompts: one for article generation and one for quiz creation.

**Article prompt** — instructs the LLM to rewrite the transcript as a professional English article with key takeaways and estimated reading time.

**Quiz prompt** — instructs the LLM to create 5-10 multiple-choice questions with 4 options each and per-option feedback explaining why each answer is correct or incorrect.

Customize these for your content type, language, or difficulty level.


In [ ]:
ARTICLE_SYSTEM_PROMPT = """You are an expert content creator.
Your goal is to provide a clean, professionally written article based on the provided transcribed text.
The article should present the content of the video in a clean professional way and be suitable for a general reader.
At the end of the article, include key takeaways or bullet points.
Also, calculate and provide the estimated time needed to read the generated article.
YOU MUST respond ONLY with valid JSON in the following format. Do not include markdown formatting or explanations outside the JSON.

{
  "article_text": "Your professionally written article summary in English with key takeaways at the end.",
  "time_to_read_minutes": 5
}"""

QUIZ_SYSTEM_PROMPT = """You are an expert educator and quiz creator.
Your goal is to generate a multiple-choice quiz based on the provided article content and supporting sentences.
The quiz should consist of 5 to 10 non-trivial multiple-choice questions.
Each question must have exactly 4 options.
For each option, provide specific feedback that explains why the option is correct or incorrect.
YOU MUST respond ONLY with valid JSON in the following format. Do not include markdown formatting or explanations outside the JSON.

{
  "quizzes": [
    {
      "question": "The quiz question.",
      "options": [
        {"text": "Option 1 text", "is_correct": true, "feedback": "Explanation why this is correct."},
        {"text": "Option 2 text", "is_correct": false, "feedback": "Explanation why this is incorrect."},
        {"text": "Option 3 text", "is_correct": false, "feedback": "Explanation why this is incorrect."},
        {"text": "Option 4 text", "is_correct": false, "feedback": "Explanation why this is incorrect."}
      ]
    }
  ]
}"""

print("System prompts ready.")


## Article Generation

For each transcript, extracts the full Arabic text and sends it to the LLM to produce a professional English article.


In [ ]:
def clean_llm_json(raw):
    """Strip markdown code fences from LLM response if present."""
    if raw.startswith("```json"):
        raw = raw[7:]
    if raw.startswith("```"):
        raw = raw[3:]
    if raw.endswith("```"):
        raw = raw[:-3]
    return raw.strip()


def generate_article(json_path):
    """Generate an article from a transcript JSON file. Returns the output path or None."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Extract the full text from the transcript
    try:
        full_text = data['transcripts'][0]['text']
    except (KeyError, IndexError):
        print(f"[skip] No 'text' field in {os.path.basename(json_path)}")
        return None

    print(f"[article] {os.path.basename(json_path)}...")
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": ARTICLE_SYSTEM_PROMPT},
            {"role": "user", "content": f"Here is the transcribed text:\n\n{full_text}"}
        ],
        temperature=0.2
    )

    raw = clean_llm_json(response.choices[0].message.content.strip())

    try:
        article_data = json.loads(raw)
        base_name = os.path.splitext(os.path.basename(json_path))[0]
        output_path = os.path.join(OUTPUT_FOLDER, f"{base_name}_article.json")
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(article_data, f, indent=4, ensure_ascii=False)
        print(f"[saved] {base_name}_article.json")
        return output_path
    except json.JSONDecodeError:
        print(f"[error] Failed to parse article JSON for {os.path.basename(json_path)}")
        print(f"  Response preview: {raw[:500]}")
        return None


## Quiz Generation

Takes each generated article plus the original transcript sentences and produces a multiple-choice quiz with detailed feedback per option.


In [ ]:
def generate_quiz(article_path, transcript_path):
    """Generate a quiz from an article + its source transcript sentences."""
    with open(article_path, 'r', encoding='utf-8') as f:
        article_data = json.load(f)
    article_text = article_data.get('article_text', '')

    with open(transcript_path, 'r', encoding='utf-8') as f:
        transcript_data = json.load(f)

    # Extract sentences from transcript
    sentences = []
    try:
        if 'transcripts' in transcript_data and transcript_data['transcripts']:
            sentences = transcript_data['transcripts'][0].get('sentences', [])
    except (KeyError, IndexError):
        pass

    if not sentences:
        print(f"[skip] No sentences found for quiz: {os.path.basename(transcript_path)}")
        return

    sentences_text = json.dumps(sentences, ensure_ascii=False)
    combined = f"Here is the article text:\n{article_text}\n\nAnd here are the supporting sentences from the transcript:\n{sentences_text}"

    print(f"[quiz] {os.path.basename(transcript_path)}...")
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": QUIZ_SYSTEM_PROMPT},
            {"role": "user", "content": combined}
        ],
        temperature=0.7  # Higher temp for more varied questions
    )

    raw = clean_llm_json(response.choices[0].message.content.strip())

    try:
        quiz_data = json.loads(raw)
        base_name = os.path.splitext(os.path.basename(transcript_path))[0]
        quiz_path = os.path.join(OUTPUT_FOLDER, f"{base_name}_quiz.json")
        with open(quiz_path, 'w', encoding='utf-8') as f:
            json.dump(quiz_data, f, indent=4, ensure_ascii=False)
        print(f"[saved] {base_name}_quiz.json")
    except json.JSONDecodeError:
        print(f"[error] Failed to parse quiz JSON for {os.path.basename(transcript_path)}")
        print(f"  Response preview: {raw[:500]}")


## Run Pipeline

Generates articles first, then quizzes. Files ending in `_article.json` or `_quiz.json` are automatically excluded from processing.


In [ ]:
# Find all transcript JSONs (exclude previously generated articles and quizzes)
transcript_files = sorted([
    f for f in glob.glob(os.path.join(INPUT_FOLDER, "*.json"))
    if not f.endswith('_article.json') and not f.endswith('_quiz.json')
])
print(f"Found {len(transcript_files)} transcript(s).\n")

# Phase 1: Generate articles
article_pairs = []  # (article_path, transcript_path)
for transcript_path in transcript_files:
    article_path = generate_article(transcript_path)
    if article_path:
        article_pairs.append((article_path, transcript_path))

# Phase 2: Generate quizzes
print(f"\n--- Quiz Generation ({len(article_pairs)} article(s)) ---\n")
for article_path, transcript_path in article_pairs:
    generate_quiz(article_path, transcript_path)

print("\nArticle and quiz generation complete.")
